In [1]:
# ЯЧЕЙКА 1: только проверка интернет-канала (не блокирует работу ноутбука).
import os  # Импорт os нужен для чтения переменных окружения (если пользователь захочет переопределить пороги).
import re  # Импорт re нужен для извлечения среднего ping из текстового вывода команды ping.
import subprocess  # Импорт subprocess нужен для запуска системной команды ping.

MIN_DOWNLOAD_MBPS = float(os.environ.get("MIN_DOWNLOAD_MBPS", "300"))  # Константа задаёт минимально желаемую скорость скачивания в Мбит/с.
MAX_PING_MS = float(os.environ.get("MAX_PING_MS", "100"))  # Константа задаёт максимально желаемый средний ping в миллисекундах.
PING_HOSTS = ["civitai.com", "huggingface.co"]  # Список сайтов для проверки ping (критерий ping остаётся прежним).


def ensure_speedtest_package():
    """Функция проверяет наличие speedtest и устанавливает пакет, если он отсутствует."""
    import importlib.util as importlib_util  # Локальный импорт для проверки существования Python-модуля.
    module_exists = importlib_util.find_spec("speedtest") is not None  # Флаг показывает, установлен ли модуль speedtest.
    if module_exists:
        print("speedtest-cli: уже установлен")
        return True
    print("speedtest-cli: не найден, устанавливаю...")
    install_proc = subprocess.run(  # Результат выполнения pip install сохраняется в переменную install_proc.
        ["python", "-m", "pip", "install", "-U", "speedtest-cli"],
        text=True,
        capture_output=True,
    )
    if install_proc.returncode != 0:  # Если код возврата не нулевой, установка не удалась.
        print("[WARN] Не удалось установить speedtest-cli. Проверка скорости будет пропущена.")
        print(install_proc.stdout[-800:])
        print(install_proc.stderr[-800:])
        return False
    return True


def run_speedtest():
    """Функция измеряет download/upload скорость и возвращает словарь результатов."""
    import speedtest  # Импорт внутри функции, чтобы не ломать весь ноутбук, если пакет не установлен.
    result = {"download_mbps": None, "upload_mbps": None, "error": None}  # Словарь result хранит результат теста сети.
    try:
        st = speedtest.Speedtest(secure=True)  # Объект st выполняет подбор сервера и сами измерения.
        st.get_best_server()
        download_bps = st.download()  # Скорость скачивания в битах в секунду.
        upload_bps = st.upload()  # Скорость отдачи в битах в секунду.
        result["download_mbps"] = download_bps / 1_000_000  # Перевод в Мбит/с.
        result["upload_mbps"] = upload_bps / 1_000_000  # Перевод в Мбит/с.
    except Exception as exc:
        result["error"] = str(exc)  # В поле error сохраняется причина сбоя speedtest.
    return result


def avg_ping(host):
    """Функция запускает ping до host и возвращает среднее значение в мс."""
    cmd = ["ping", "-c", "3", "-W", "2", host]  # Команда ping: 3 пакета и таймаут 2 секунды на ответ.
    try:
        ping_proc = subprocess.run(cmd, text=True, capture_output=True)  # Результат ping сохраняем в ping_proc.
    except FileNotFoundError:
        return None, "Команда ping недоступна в этом окружении"
    if ping_proc.returncode != 0:
        return None, ping_proc.stderr.strip() or ping_proc.stdout.strip()  # Возвращаем ошибку, если ping неуспешен.
    summary_match = re.search(r"=\s*([\d\.]+)/([\d\.]+)/([\d\.]+)/", ping_proc.stdout)  # Поиск строки min/avg/max.
    if not summary_match:
        return None, "Не удалось распарсить средний ping"
    avg_ms = float(summary_match.group(2))  # Второе число в шаблоне — это средний ping (avg).
    return avg_ms, None


print("Проверка интернета (требования: download >= 300 Мбит/с и ping < 100 мс)...")
can_run_speedtest = ensure_speedtest_package()  # Флаг показывает, доступен ли speedtest для измерения скорости.
net = {"download_mbps": None, "upload_mbps": None, "pings": {}}  # Словарь net хранит итоговые сетевые метрики.

if can_run_speedtest:
    speed_result = run_speedtest()  # Переменная speed_result хранит результат speedtest.
    net["download_mbps"] = speed_result.get("download_mbps")
    net["upload_mbps"] = speed_result.get("upload_mbps")
    if speed_result.get("error"):
        print(f"[WARN] Ошибка speedtest: {speed_result['error']}")
else:
    print("[WARN] Speedtest пропущен (модуль недоступен).")

for host in PING_HOSTS:
    host_ping, host_err = avg_ping(host)  # host_ping — средний ping, host_err — текст ошибки.
    net["pings"][host] = {"avg_ms": host_ping, "error": host_err}

print("\n--- Результаты сети ---")
print(f"Download: {net['download_mbps']:.2f} Мбит/с" if net["download_mbps"] is not None else "Download: N/A")
print(f"Upload:   {net['upload_mbps']:.2f} Мбит/с" if net["upload_mbps"] is not None else "Upload:   N/A")
for host, host_data in net["pings"].items():
    print(f"Ping {host}: {host_data['avg_ms']:.2f} мс" if host_data["avg_ms"] is not None else f"Ping {host}: N/A ({host_data['error']})")

speed_ok = net["download_mbps"] is not None and net["download_mbps"] >= MIN_DOWNLOAD_MBPS  # Проверка нового критерия скорости (>=300).
ping_ok = all(host_data["avg_ms"] is not None and host_data["avg_ms"] < MAX_PING_MS for host_data in net["pings"].values())  # Проверка старого критерия ping.

if speed_ok and ping_ok:
    print("Сеть соответствует требованиям для загрузок и работы.")
else:
    print("Предупреждение: сеть ниже рекомендуемого уровня (download < 300 Мбит/с и/или ping >= 100 мс).")


Проверка интернета (требования: download >= 300 Мбит/с и ping < 100 мс)...
speedtest-cli: не найден, устанавливаю...

--- Результаты сети ---
Download: 4581.23 Мбит/с
Upload:   807.03 Мбит/с
Ping civitai.com: N/A (Команда ping недоступна в этом окружении)
Ping huggingface.co: N/A (Команда ping недоступна в этом окружении)
Предупреждение: сеть ниже рекомендуемого уровня (download < 300 Мбит/с и/или ping >= 100 мс).


In [2]:
# ЯЧЕЙКА 2: базовые переменные окружения и директории только для ComfyUI на Vast.ai.
import os  # Импорт os нужен для работы с переменными окружения.
from pathlib import Path  # Импорт Path нужен для понятной и безопасной работы с путями.
from getpass import getpass #Импорт для ручного ввода

ON_VAST = any(key in os.environ for key in ("VAST_CONTAINERLABEL", "VAST_TCP_PORT_22", "CONTAINER_ID")) or Path("/workspace").exists()  # Флаг окружения Vast.ai.
if not ON_VAST:
    print("Внимание: окружение не похоже на Vast.ai, но код продолжит работу.")

BASE_DIR = Path(os.environ.get("BASE_DIR", "/workspace"))  # Корневая рабочая директория.
COMFY_DIR = BASE_DIR / "ComfyUI"  # Путь к репозиторию ComfyUI.
VOLUME_DIR = Path(os.environ.get("VOLUME_DIR", str(BASE_DIR / "volume")))  # Путь к постоянному тому (volume).

MODELS_DIR = COMFY_DIR / "models"  # Общая папка моделей ComfyUI.
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"  # Папка базовых моделей (checkpoints).
LORA_DIR = MODELS_DIR / "loras"  # Папка LoRA моделей.
UPSCALER_DIR = MODELS_DIR / "upscale_models"  # Папка upscaler моделей.
VAE_DIR = MODELS_DIR / "vae"  # Папка VAE.
CONTROLNET_DIR = MODELS_DIR / "controlnet"  # Папка ControlNet моделей.
IPADAPTER_DIR = MODELS_DIR / "ipadapter"  # Папка IP-Adapter моделей.
CUSTOM_NODES_DIR = COMFY_DIR / "custom_nodes"  # Папка для custom nodes ComfyUI.

SECRET_ENV_NAMES = {  # Словарь для отображения наличия секретов или ввода).
    "CIVITAI_TOKEN": os.environ.get("CIVITAI_TOKEN") or getpass("Введите CIVITAI_TOKEN: "),
    "HF_TOKEN": os.environ.get("HF_TOKEN") or getpass("Введите HF_TOKEN: "),
    "GRADIO_TOKEN": os.environ.get("GRADIO_TOKEN") or getpass("Введите GRADIO_TOKEN: "),
}
for required_dir in [BASE_DIR, COMFY_DIR]:
    required_dir.mkdir(parents=True, exist_ok=True)  # Создаём директории заранее, чтобы следующие ячейки работали стабильно.

print("Базовые переменные и директории ComfyUI подготовлены.")
print(f"COMFY_DIR: {COMFY_DIR}")
print(f"VOLUME_DIR: {VOLUME_DIR}")


Внимание: окружение не похоже на Vast.ai, но код продолжит работу.
Введите CIVITAI_TOKEN: ··········
Введите HF_TOKEN: ··········
Введите GRADIO_TOKEN: ··········
Базовые переменные и директории ComfyUI подготовлены.
COMFY_DIR: /workspace/ComfyUI
VOLUME_DIR: /workspace/volume


In [3]:
# ЯЧЕЙКА 3: корректный порядок — сначала ComfyUI, потом зависимости, потом создание остальных директорий.
import shutil  # Импорт shutil нужен для удаления конфликтной директории ComfyUI без создания backup-папок.
import subprocess  # Импорт subprocess нужен для запуска apt/pip/git команд.
from pathlib import Path  # Импорт Path нужен для проверки наличия папок/файлов.

COMFY_REPO_URL = "https://github.com/comfyanonymous/ComfyUI.git"  # URL официального репозитория ComfyUI.
APT_PACKAGES_AFTER_CLONE = ["python3-venv", "python3-pip", "aria2", "libgl1", "libglib2.0-0"]  # Системные пакеты, которые ставим после получения ComfyUI.
PYTHON_PACKAGES = ["tqdm", "huggingface_hub", "gdown"]  # Базовые Python-пакеты для последующих ячеек загрузки.
IPADAPTER_PLUS_PACKAGES = ["insightface", "onnxruntime", "opencv-python-headless"]  # Пакеты для корректной работы ComfyUI_IPAdapter_plus.


def run_cmd(cmd, check=True):
    """Функция запускает системную команду, печатает её и возвращает объект результата."""
    print("[CMD]", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=True)


def run_cmd_with_logs(cmd, check=False):
    """Функция запускает команду и печатает последние строки stdout/stderr для диагностики."""
    result = run_cmd(cmd, check=check)
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    return result


def ensure_comfy_repo():
    """Функция гарантирует наличие актуального git-репозитория ComfyUI в COMFY_DIR без backup-папок."""
    comfy_path = Path(COMFY_DIR)  # comfy_path — целевой путь репозитория ComfyUI.
    comfy_git_dir = comfy_path / ".git"  # comfy_git_dir — путь к .git внутри COMFY_DIR.

    if comfy_git_dir.exists():
        print("ComfyUI уже git-репозиторий, выполняю git pull...")
        run_cmd_with_logs(["git", "-C", str(comfy_path), "pull", "--ff-only"], check=False)
        return

    if comfy_path.exists() and any(comfy_path.iterdir()):
        print(f"Найдена непустая не-git папка: {comfy_path}")
        print("Удаляю её без создания backup, чтобы clone прошёл в чистую директорию.")
        shutil.rmtree(comfy_path)

    print("ComfyUI не найден, выполняю git clone...")
    clone_result = run_cmd_with_logs(["git", "clone", COMFY_REPO_URL, str(comfy_path)], check=False)
    if clone_result.returncode != 0:
        raise RuntimeError("Не удалось клонировать ComfyUI. Смотрите stderr выше (частая причина: сеть/доступ к GitHub).")


run_cmd_with_logs(["apt", "update", "-qq"], check=False)
run_cmd_with_logs(["apt", "install", "-y", "-qq", "git"], check=False)  # Ставим git до clone/pull.
ensure_comfy_repo()  # 1) Сначала гарантируем наличие ComfyUI.

run_cmd_with_logs(["apt", "install", "-y", "-qq", *APT_PACKAGES_AFTER_CLONE], check=False)  # 2) Ставим остальные системные зависимости.
run_cmd_with_logs(["python", "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=False)
run_cmd_with_logs(["python", "-m", "pip", "install", *PYTHON_PACKAGES], check=False)  # Ставим базовые пакеты без форс-апдейта всего окружения.
run_cmd_with_logs(["python", "-m", "pip", "install", *IPADAPTER_PLUS_PACKAGES], check=False)  # Ставим зависимости IPAdapter_plus.

requirements_file = Path(COMFY_DIR) / "requirements.txt"  # requirements_file — путь к requirements ComfyUI.
if requirements_file.exists():
    run_cmd_with_logs(["python", "-m", "pip", "install", "-r", str(requirements_file)], check=False)

# 3) После clone/install создаём рабочие папки (кроме BASE_DIR и COMFY_DIR, они уже существуют/управляются отдельно).
for required_dir in [VOLUME_DIR, MODELS_DIR, CHECKPOINTS_DIR, LORA_DIR, UPSCALER_DIR, VAE_DIR, CONTROLNET_DIR, IPADAPTER_DIR, CUSTOM_NODES_DIR]:
    Path(required_dir).mkdir(parents=True, exist_ok=True)

print("Ячейка 3 завершена: ComfyUI, зависимости и директории готовы.")

[CMD] apt update -qq
88 packages can be upgraded. Run 'apt list --upgradable' to see them.



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

[CMD] apt install -y -qq git
Suggested packages:
  gettext-base git-daemon-run | git-daemon-sysvinit git-doc git-email git-gui
  gitk gitweb git-cvs git-mediawiki git-svn
The following packages will be upgraded:
  git
1 upgraded, 0 newly installed, 0 to remove and 87 not upgraded.
Need to get 3,172 kB of archives.
After this operation, 4,096 B of additional disk space will be used.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 6

In [4]:
# ЯЧЕЙКА 4: формируем списки и выполняем параллельные загрузки (до 5 потоков, приоритет: тяжёлые -> лёгкие).
import os  # Импорт os нужен для чтения токенов и настроек из переменных окружения.
import subprocess  # Импорт subprocess нужен для git/aria2c команд.
from pathlib import Path  # Импорт Path нужен для работы с путями и создания папок.
from concurrent.futures import ThreadPoolExecutor, as_completed  # Импорт для параллельных скачиваний.

MAX_PARALLEL_DOWNLOADS = int(os.environ.get("MAX_PARALLEL_DOWNLOADS", "5"))  # Максимум параллельных загрузок (расширено до 5).
MIN_VALID_FILE_BYTES = int(os.environ.get("MIN_VALID_FILE_BYTES", "1024"))  # Минимальный валидный размер файла.
INVALID_TEXT_BYTES_LIMIT = int(os.environ.get("INVALID_TEXT_BYTES_LIMIT", str(5 * 1024 * 1024)))  # До какого размера файл проверяем как потенциально текстовую ошибку.
FORCE_REDOWNLOAD = os.environ.get("FORCE_REDOWNLOAD", "0").lower() in {"1", "true", "yes", "y"}  # Флаг принудительного перекачивания.
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")  # Токен Civitai для приватных/ограниченных ссылок.
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Токен Hugging Face для gated/ограниченных ссылок.

CUSTOM_NODE_REPOS = [  # Список custom nodes, обязательных по ТЗ.
    "https://github.com/ltdrdata/ComfyUI-Manager",
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus",
    "https://github.com/Fannovel16/comfyui_controlnet_aux",
]
# Список URL базовых моделей (checkpoints). Добавляйте ваши ссылки.
CHECKPOINT_URLS = [
    #WAI
    "https://civitai.com/api/download/models/2514310?type=Model&format=SafeTensor&size=pruned&fp=fp16",
    #Pony
    "https://civitai.com/api/download/models/290640",
    #Juggernaut XL
    "https://civitai.com/api/download/models/1759168",
]
 # Список URL LoRA моделей.
LORA_URLS = [
    #Illustrious LoRA
    #Detailer IL V2 218 MB
    "https://civitai.com/api/download/models/1736373?type=Model&format=SafeTensor",
    #Realistic filter V1 55 MB
    "https://civitai.com/api/download/models/1124771?type=Model&format=SafeTensor",
    #Hyperrealistic V4 ILL 435 MB
    "https://civitai.com/api/download/models/1914557?type=Model&format=SafeTensor",
    #Niji semi realism V3.5 ILL 435 MB
    "https://civitai.com/api/download/models/1882710?type=Model&format=SafeTensor",
    #ATNR Style ILL V1.1 350 MB
    "https://civitai.com/api/download/models/1711464?type=Model&format=SafeTensor",
    #Face Enhancer Ill 218 MB
    "https://civitai.com/api/download/models/1839268?type=Model&format=SafeTensor",
    #Smooth Detailer Booster V4 243 MB
    "https://civitai.com/api/download/models/2196453?type=Model&format=SafeTensor",
    #USNR Style V-pred 157 MB
    "https://civitai.com/api/download/models/2555444?type=Model&format=SafeTensor",
    #748cm Style V1 243 MB",
    "https://civitai.com/api/download/models/1056404?type=Model&format=SafeTensor",
    #Velvet's Mythic Fantasy Styles IL 218 MB
    "https://civitai.com/api/download/models/2620790?type=Model&format=SafeTensor",
    #Pixel Art Style IL V7 435 MB
    "https://civitai.com/api/download/models/2661972?type=Model&format=SafeTensor",
    #Pony LoRA
    #Abstract Painting Style Pony V6 XL 54,8 MB
    "https://civitai.com/api/download/models/1558543",
    #incase style v3 1 ponyxl ilff 243 MB
    "https://civitai.com/api/download/models/436219",
    #CHP pony 218 MB
    "https://civitai.com/api/download/models/1091750",
    #vixons pony styles 218 MB
    "https://civitai.com/api/download/models/1029540",
    #Realistic AnimeIXL v2 218 MB
    "https://civitai.com/api/download/models/1169083",
    #Popyay's Epic Fantasy Style 218 MB
    "https://civitai.com/api/download/models/522995",
]
# Список URL upscaler моделей.
UPSCALER_URLS = [
    #WAI default Upscaler
    #Real-ESRGAN-x4plus
    "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    #Pony default Upscaler
    #4x-UltraSharp 64 MB
    "https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth",
    #Juggernaut default Upscaler
    #4x_NMKD-Siax_200k 64 MB
    "https://huggingface.co/uwg/upscaler/resolve/main/ESRGAN/4x_NMKD-Siax_200k.pth",
]
# Список URL VAE файлов.
VAE_URLS = [
    #Pony default VAE
    "https://civitai.com/api/download/models/290640?type=VAE&format=SafeTensor",
]
# Список URL ControlNet моделей.
CONTROLNET_URLS = [
    #t2i-adapter_xl_openpose 151 MB
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/t2i-adapter_xl_openpose.safetensors",
    #t2i-adapter_xl_canny 148 MB
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/t2i-adapter_xl_canny.safetensors",
    #t2i-adapter_xl_sketch 148 MB
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/t2i-adapter_xl_sketch.safetensors",
    #t2i-adapter_diffusers_xl_depth_midas 151 MB
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/t2i-adapter_diffusers_xl_depth_midas.safetensors",
    #t2i-adapter_diffusers_xl_depth_zoe 151 MB
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/t2i-adapter_diffusers_xl_depth_zoe.safetensors",
    #t2i-adapter_diffusers_xl_lineart 151 MB
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/t2i-adapter_diffusers_xl_lineart.safetensors",
    #diffusers_xl_canny_full 2,5 GB"
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_canny_full.safetensors",
    #diffusers_xl_depth_full 2,5 GB
    "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_depth_full.safetensors",
    #thibaud_xl_openpose 2,5 GB"
     "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/thibaud_xl_openpose.safetensors",
    #xinsir_union_sdxl 2,5 GB
    "https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model_promax.safetensors",
    #xinsir_openpose_sdxl 2,5 GB
    "https://huggingface.co/xinsir/controlnet-openpose-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors",
]
# Список URL IPAdapter моделей.
IPADAPTER_URLS = [
    "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin",
]


def run_cmd(cmd, check=False):
    """Функция запускает команду и печатает её для прозрачности выполнения."""
    print("[CMD]", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, check=check)


def with_token_if_needed(url):
    """Функция добавляет токен в URL для HF/Civitai, если токен есть и он ещё не добавлен."""
    if "huggingface.co" in url and HF_TOKEN and "token=" not in url:
        separator = "&" if "?" in url else "?"
        return f"{url}{separator}token={HF_TOKEN}"
    if "civitai.com" in url and CIVITAI_TOKEN and "token=" not in url:
        separator = "&" if "?" in url else "?"
        return f"{url}{separator}token={CIVITAI_TOKEN}"
    return url


def file_name_from_url(url):
    """Функция извлекает имя файла из URL-адреса."""
    return url.split("?")[0].rstrip("/").split("/")[-1]


def ensure_custom_node(repo_url):
    """Функция клонирует custom node или обновляет его, если репозиторий уже существует."""
    repo_name = repo_url.rstrip("/").split("/")[-1]
    repo_dir = Path(CUSTOM_NODES_DIR) / repo_name
    if (repo_dir / ".git").exists():
        run_cmd(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
    else:
        run_cmd(["git", "clone", repo_url, str(repo_dir)], check=False)


def is_probably_text_error_file(file_path):
    """Функция проверяет, похож ли файл на HTML/JSON с ошибкой вместо модели."""
    file_path = Path(file_path)
    if not file_path.exists() or file_path.stat().st_size > INVALID_TEXT_BYTES_LIMIT:
        return False
    with open(file_path, "rb") as fh:
        raw = fh.read(4096)
    sample = raw.decode("utf-8", errors="ignore").lower()
    markers = ["<html", "<!doctype html", "access denied", "error", "unauthorized", "forbidden", '"error"', "{" ]
    return any(marker in sample for marker in markers)


def download_one(url, target_dir):
    """Функция скачивает один файл в нужную папку и возвращает статус операции."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    safe_url = with_token_if_needed(url)
    file_name = file_name_from_url(url)
    output_path = target_dir / file_name

    if output_path.exists() and output_path.stat().st_size >= MIN_VALID_FILE_BYTES and not FORCE_REDOWNLOAD:
        if is_probably_text_error_file(output_path):
            print(f"[RETRY] {output_path} выглядит как текстовая ошибка, перекачиваю")
            output_path.unlink(missing_ok=True)
        else:
            print(f"[SKIP] {output_path} уже существует ({output_path.stat().st_size} bytes)")
            return (url, "skip", str(output_path))

    if FORCE_REDOWNLOAD and output_path.exists():
        print(f"[RETRY] FORCE_REDOWNLOAD=1, удаляю {output_path}")
        output_path.unlink(missing_ok=True)

    run_cmd([
        "aria2c", safe_url,
        "--follow-torrent=false",
        "--max-connection-per-server=8",
        "--split=8",
        "--min-split-size=1M",
        "--allow-overwrite=true",
        "--auto-file-renaming=false",
        "-d", str(target_dir),
        "-o", file_name,
    ], check=False)

    if output_path.exists() and output_path.stat().st_size >= MIN_VALID_FILE_BYTES and not is_probably_text_error_file(output_path):
        return (url, "ok", str(output_path))

    print(f"[FAIL] {url} -> {output_path} (файл отсутствует или похож на ошибку)")
    return (url, "fail", str(output_path))


for repo in CUSTOM_NODE_REPOS:
    ensure_custom_node(repo)


download_jobs = []
for url in CHECKPOINT_URLS:
    download_jobs.append((url, CHECKPOINTS_DIR, 1))
for url in CONTROLNET_URLS:
    download_jobs.append((url, CONTROLNET_DIR, 1))
for url in IPADAPTER_URLS:
    download_jobs.append((url, IPADAPTER_DIR, 1))
for url in LORA_URLS:
    download_jobs.append((url, LORA_DIR, 2))
for url in VAE_URLS:
    download_jobs.append((url, VAE_DIR, 3))
for url in UPSCALER_URLS:
    download_jobs.append((url, UPSCALER_DIR, 4))

download_jobs = sorted(download_jobs, key=lambda item: item[2])
results = []

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_DOWNLOADS) as pool:
    future_map = {pool.submit(download_one, job[0], job[1]): job for job in download_jobs}
    for future in as_completed(future_map):
        results.append(future.result())

ok_count = sum(1 for _, status, _ in results if status == "ok")
skip_count = sum(1 for _, status, _ in results if status == "skip")
fail_count = sum(1 for _, status, _ in results if status == "fail")
print(f"Загрузки завершены: ok={ok_count}, skip={skip_count}, fail={fail_count}")
if fail_count:
    print("Есть неуспешные загрузки, проверьте ссылки/токены. При необходимости включите FORCE_REDOWNLOAD=1")


def install_repo_requirements(repo_dir):
    """Функция ставит requirements custom node репозитория."""
    req_file = Path(repo_dir) / "requirements.txt"
    if req_file.exists():
        run_cmd(["python", "-m", "pip", "install", "-U", "-r", str(req_file)], check=False)


def ensure_custom_node(repo_url):
    """Функция клонирует custom node или обновляет его, если репозиторий уже существует."""
    repo_name = repo_url.rstrip("/").split("/")[-1]
    repo_dir = Path(CUSTOM_NODES_DIR) / repo_name
    if (repo_dir / ".git").exists():
        run_cmd(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
    else:
        run_cmd(["git", "clone", repo_url, str(repo_dir)], check=False)
    install_repo_requirements(repo_dir)


[CMD] git clone https://github.com/ltdrdata/ComfyUI-Manager /workspace/ComfyUI/custom_nodes/ComfyUI-Manager
[CMD] git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus /workspace/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
[CMD] git clone https://github.com/Fannovel16/comfyui_controlnet_aux /workspace/ComfyUI/custom_nodes/comfyui_controlnet_aux
[CMD] aria2c https://civitai.com/api/download/models/2514310?type=Model&format=SafeTensor&size=pruned&fp=fp16 --follow-torrent=false --max-connection-per-server=8 --split=8 --min-split-size=1M --allow-overwrite=true --auto-file-renaming=false -d /workspace/ComfyUI/models/checkpoints -o 2514310
[CMD] aria2c https://civitai.com/api/download/models/1759168 --follow-torrent=false --max-connection-per-server=8 --split=8 --min-split-size=1M --allow-overwrite=true --auto-file-renaming=false -d /workspace/ComfyUI/models/checkpoints -o 1759168
[CMD] aria2c https://civitai.com/api/download/models/290640 --follow-torrent=false --max-connection-per-serv

In [5]:
# ЯЧЕЙКА 5: запуск ComfyUI в двух режимах по выбору пользователя.
import os  # Импорт os нужен для токена и переменных окружения.
import re  # Импорт re нужен для поиска публичного URL в логах cloudflared.
import select  # Импорт select нужен для неблокирующего чтения stdout процесса.
import subprocess  # Импорт subprocess нужен для запуска процессов ComfyUI/cloudflared.
import time  # Импорт time нужен для задержек при ожидании URL туннеля.
import urllib.request  # Импорт urllib нужен для health-check HTTP порта ComfyUI.
from pathlib import Path  # Импорт Path нужен для проверки существования main.py.


comfy_main = Path(COMFY_DIR) / "main.py"
if not comfy_main.exists():
    raise FileNotFoundError(f"Не найден файл запуска ComfyUI: {comfy_main}")


def wait_comfy_ready(comfy_proc, timeout_sec=90):
    """Ждём доступности ComfyUI и параллельно печатаем логи старта."""
    start = time.time()
    health_url = "http://127.0.0.1:8188/system_stats"
    while time.time() - start < timeout_sec:
        if comfy_proc.poll() is not None:
            print(f"[ComfyUI] Процесс завершился с кодом {comfy_proc.returncode}")
            return False
        ready_to_read, _, _ = select.select([comfy_proc.stdout], [], [], 0.5)
        if ready_to_read:
            line = comfy_proc.stdout.readline()
            if line:
                print(f"[ComfyUI] {line.rstrip()}")
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                if response.status == 200:
                    print("[ComfyUI] Health-check OK: UI запущен и отвечает на /system_stats")
                    return True
        except Exception:
            pass
    print("[ComfyUI] Таймаут ожидания запуска. Проверьте логи выше.")
    return False


print("1) вариант запуска: локальный режим без доступа извне (для SSH-туннеля на Vast.ai)")
print("2) вариант запуска: временный сайт через tunnel + GRADIO_AUTH (или GRADIO_TOKEN для обратной совместимости)")
launch_mode = input("Введите режим запуска (1/2): ").strip()
if launch_mode not in {"1", "2"}:
    raise ValueError("Нужно ввести только 1 или 2")

comfy_cmd = ["python", str(comfy_main), "--port", "8188"]

if launch_mode == "1":
    comfy_cmd += ["--listen", "127.0.0.1"]
    print("Запуск локального режима ComfyUI. Используйте SSH-туннель к 127.0.0.1:8188")
    comfy_proc = subprocess.Popen(comfy_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    wait_comfy_ready(comfy_proc)
else:
    comfy_cmd += ["--listen", "0.0.0.0"]
    print("Запуск ComfyUI для временного внешнего доступа...")
    comfy_proc = subprocess.Popen(comfy_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    is_ready = wait_comfy_ready(comfy_proc)

    cloudflared_exists = subprocess.run(["bash", "-lc", "command -v cloudflared >/dev/null"], check=False).returncode == 0
    if not cloudflared_exists:
        subprocess.run([
            "bash", "-lc",
            "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"
        ], check=False)

    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    public_url = None
    start_time = time.time()
    while time.time() - start_time < 45:
        line = tunnel_proc.stdout.readline()
        if not line:
            continue
        print(f"[cloudflared] {line.rstrip()}")
        match = re.search(r"https://[a-zA-Z0-9\-\.]+trycloudflare.com", line)
        if match:
            public_url = match.group(0)
            break

    access_auth = os.environ.get("GRADIO_AUTH", os.environ.get("GRADIO_TOKEN", ""))
    if public_url:
        print("Временный публичный URL создан:")
        print(public_url)
        if access_auth:
            if ":" in access_auth:
                print("GRADIO_AUTH обнаружен в формате login:password.")
                print("Можно пробовать basic-auth URL (если клиент/браузер поддерживает):")
                print(public_url.replace("https://", f"https://{access_auth}@", 1))
            else:
                print("Передан токен без ':', добавляю как query-параметр для совместимости:")
                print(f"{public_url}/?token={access_auth}")
        else:
            print("GRADIO_AUTH не задан. Доступ без токена.")
        if not is_ready:
            print("Внимание: tunnel поднят, но health-check ComfyUI не подтвердил запуск.")
    else:
        print("Не удалось получить публичный URL tunnel за отведённое время.")


1) вариант запуска: локальный режим без доступа извне (для SSH-туннеля на Vast.ai)
2) вариант запуска: временный сайт через tunnel + GRADIO_AUTH (или GRADIO_TOKEN для обратной совместимости)
Введите режим запуска (1/2): 2
Запуск ComfyUI для временного внешнего доступа...
[ComfyUI] [START] Security scan
[ComfyUI] [DONE] Security scan
[ComfyUI] ## ComfyUI-Manager: installing dependencies done.
[ComfyUI] ** ComfyUI startup time: 2026-02-27 19:55:54.656
[ComfyUI] ** Platform: Linux
[ComfyUI] ** Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
[ComfyUI] ** Python executable: /usr/bin/python3
[ComfyUI] ** ComfyUI Path: /workspace/ComfyUI
[ComfyUI] ** ComfyUI Base Folder Path: /workspace/ComfyUI
[ComfyUI] ** User directory: /workspace/ComfyUI/user
[ComfyUI] ** ComfyUI-Manager config path: /workspace/ComfyUI/user/__manager/config.ini
[ComfyUI] ** Log path: /workspace/ComfyUI/user/comfyui.log
[ComfyUI] 
[ComfyUI] Prestartup times for custom nodes:
[ComfyUI]    4.9 seconds: /wo

In [6]:
# ЯЧЕЙКА 6: остановка (убийство) процессов ComfyUI.
import subprocess  # Импорт subprocess нужен для выполнения команды завершения процесса.

kill_cmd = ["bash", "-lc", "pkill -f 'ComfyUI/main.py' || true"]  # Команда завершает процессы, где есть путь ComfyUI/main.py.
subprocess.run(kill_cmd, check=False)  # Выполняем остановку; если процесса нет, ошибка подавляется через `|| true`.
print("Процессы ComfyUI остановлены (если были запущены).")


Процессы ComfyUI остановлены (если были запущены).
